In [ ]:
# uv add langchain-community langchain-text-splitters

# uv add faiss-cpu

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:5])

sk-sv


#### RAG 파이프 라인
* Load Data - Text Split - Indexing - Retrieval - Generation

In [2]:
# 1. ChatOpenAI와 OpenAIEmbeddings는 유지 (langchain-openai 패키지 사용)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 2. FAISS는 community 패키지로 이동
from langchain_community.vectorstores import FAISS

# 3. TextLoader도 community 패키지로 이동
from langchain_community.document_loaders import TextLoader

# 4. TextSplitter는 핵심 기능이므로 langchain-text-splitters 패키지로 분리됨
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2️. Text Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)
#print(split_docs)

# 3️. Indexing (벡터 저장)
vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())
# 로컬 파일로 저장
vectorstore.save_local("faiss_index")

# 4️. Retrieval (유사 문서 검색)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
# **질문(쿼리)**에 대해 유사한 문서를 검색하는 역할
retrieved_docs = retriever.invoke("소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
#print(retrieved_docs)

# 5️. Generation (LLM 응답 생성)
llm = ChatOpenAI(model="gpt-4o-mini")
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
#print(context)

response_context = llm.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요? 관련 정보: {context}")
print('context 적용한 결과')
pprint(response_context.content)

response = llm.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print('context 적용하지 않은 결과')
pprint(response.content)



context 적용한 결과
('소득세법 제12조에 따른 비과세소득은 다음과 같은 소득으로 구성되어 있습니다:\n'
 '\n'
 '1. **공익신탁법에 따른 공익신탁의 이익**: 공익신탁에서 발생하는 소득은 소득세를 부과하지 않습니다.\n'
 '\n'
 '2. **사업소득 중 특정 소득**:\n'
 '   - 가. 논과 밭을 작물 생산에 이용하여 발생하는 소득.\n'
 '   - 나. 주택을 1개만 소유하고 있는 자의 주택임대소득(단, 기준시가가 12억원을 초과하는 주택의 임대소득이나 국외에 있는 주택의 '
 '임대소득은 제외). 또한, 총수입금액이 대통령령으로 정한 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 발생한 소득에 '
 '한정).\n'
 '   - 다. 대통령령으로 정하는 농어가부업소득.\n'
 '   - 라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득.\n'
 '   - 마. 조림기간 5년 이상인 임지의 임목의 벌채 또는 양도로 발생하는 소득으로 연 600만원 이하의 금액.\n'
 '   - 바. 대통령령으로 정하는 작물재배업에서 발생하는 소득.\n'
 '\n'
 '이 외에도 비과세소득에 해당하는 사항들은 법령의 개정 시 변화할 수 있으니, 최신 법령을 참조하는 것이 중요합니다.')
context 적용하지 않은 결과
('소득세법에서 비과세소득에 해당하는 소득은 일반적으로 과세 대상에서 제외되는 소득을 말합니다. 비과세소득의 예시는 다음과 같습니다:\n'
 '\n'
 '1. **국가 또는 지방자치단체로부터 받은 보조금**: 특정 조건을 충족하는 경우 비과세가 될 수 있습니다.\n'
 '2. **연금소득 중 일부**: 일정한 조건을 충족하는 국민연금이나 공무원연금 등은 비과세 소득으로 간주됩니다.\n'
 '3. **재해로 인한 보상금**: 자연재해나 사고로 인한 보상금은 비과세로 처리됩니다.\n'
 '4. **대학 등에서 받는 장학금**: 특정 조건을 충족하는 장학금은 비과세 소득입니다.\n'
 '5. **사망보험금**: 

### 개선한 Source

In [3]:

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 2. FAISS는 community 패키지로 이동
from langchain_community.vectorstores import FAISS

# 3. TextLoader도 community 패키지로 이동
from langchain_community.document_loaders import TextLoader

# 4. TextSplitter는 핵심 기능이므로 langchain-text-splitters 패키지로 분리됨
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pprint import pprint

# 1. 데이터 로드 (기존과 동일)
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2. 텍스트 분할 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 크기 증가
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # 자연스러운 분할을 위한 구분자
    length_function=len,
    is_separator_regex=False,
)
split_docs = splitter.split_documents(documents)

# 3. 인덱싱 (벡터 저장)
vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())
vectorstore.save_local("faiss_index")

# 4. 검색 개선
retriever = vectorstore.as_retriever(
    search_type="mmr",  # 최대 다양성 검색
    search_kwargs={"k": 5, "fetch_k": 10}  # 더 많은 결과 검색
)

# 5. 프롬프트 엔지니어링
def generate_prompt(query, context):
    return f"""다음은 소득세법 비과세소득 관련 조항입니다. 문맥을 고려하여 질문에 답변하세요.

[관련 조항]
{context}

[질문]
{query}

[답변 요구사항]
- 비과세소득 유형을 계층적으로 구분하여 설명
- 각 항목별 구체적인 조건 명시
- 법조문의 항, 호, 목 번호를 포함
- 500자 이내로 간결하게 요약"""

# 검색 및 응답 생성
query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # 창의성 낮춤
response = llm.invoke(generate_prompt(query, context))

print('개선된 결과:')
pprint(response.content)

개선된 결과:
('소득세법에서 비과세소득은 다음과 같이 구분됩니다.\n'
 '\n'
 '1. **공익신탁 소득 (제12조 제1호)**  \n'
 '   - 「공익신탁법」에 따른 공익신탁의 이익.\n'
 '\n'
 '2. **사업소득 (제12조 제2호)**  \n'
 '   - 가. 논ㆍ밭에서 발생하는 소득.  \n'
 '   - 나. 1개의 주택 소유자의 주택임대소득(기준시가 12억원 초과 주택 제외).  \n'
 '   - 다. 대통령령으로 정하는 농어가부업소득.  \n'
 '   - 라. 대통령령으로 정하는 전통주 제조 소득.  \n'
 '   - 마. 조림기간 5년 이상 임목의 벌채 소득(연 600만원 이하).  \n'
 '   - 바. 대통령령으로 정하는 작물재배업 소득.  \n'
 '   - 사. 대통령령으로 정하는 어로어업 또는 양식어업 소득.\n'
 '\n'
 '3. **근로소득 및 퇴직소득 (제12조 제3호)**  \n'
 '   - 거. 국외 또는 북한지역 근로 급여.  \n'
 '   - 너. 국가 또는 지방자치단체의 보험료 부담.  \n'
 '   - 더. 대통령령으로 정하는 연장ㆍ야간ㆍ휴일 근로 급여.  \n'
 '   - 러. 사내급식 등 제공받는 식사대(월 20만원 이하).  \n'
 '   - 머. 출산 및 6세 이하 자녀 보육 관련 급여(월 20만원 이내).  \n'
 '   - 버. 국군포로 보수 및 퇴직일시금.  \n'
 '   - 서. 대학생 근로 대가 장학금.  \n'
 '   - 어. 직무발명 보상금(특수관계자 제외).\n'
 '\n'
 '4. **기타소득 (제12조 제5호)**  \n'
 '   - 가. 보훈급여금 및 정착금.  \n'
 '   - 나. 국가보안법에 따른 상금.  \n'
 '   - 다. 상훈법에 따른 부상 및 상금.  \n'
 '   - 라. 직무발명 보상금(특수관계자 제외).  \n'
 '   - 마. 국군포로 위로지원금.  \n'
 '   - 바. 문화유산 양도로 발생하는 소득.  \n'
 '   -